In [ ]:
from pathlib import Path
import yaml

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from mdx2.io import loadobj
from mdx2.report.map_statistics import prepare_table

# load the site config
with open("site_config.yaml") as f:
    config = yaml.safe_load(f)

# project directory
project_directory = Path(config['project_directory'])

## Map correlations comparing first 360 and second 360 

In [ ]:
# load data

DATASETS = {
    'CHESS 2025-11-05': ['chess_20251105/mac1_merge_lo', 'chess_20251105/mac1_merge_hi'],
    'CHESS 2025-09-24':['chess_20250924/mac1_merge_lo', 'chess_20250924/mac1_merge_hi'],
    'ALS 2025-10-15': ['als_20251015/mac1_merge_lo', 'als_20251015/mac1_merge_hi'],
    'ALS 2025-10-15 0.1s': ['als_20251015/mac1_7_merge_lo', 'als_20251015/mac1_7_merge_hi'],
    'ALS 2025-11-25': ['als_20251125/mac1_merge_lo', 'als_20251125/mac1_merge_hi'],
}

data = {}

# load all the merged data
for dset_key, dset_roots in DATASETS.items():
    for i, dset_root in enumerate(dset_roots):
        merged_nxs = project_directory / dset_root / 'mdx2/merged.nxs'
        crystal = loadobj(merged_nxs, 'crystal')
        symmetry =  loadobj(merged_nxs, 'symmetry')
        hkl_table =  loadobj(merged_nxs, 'hkl_table')
        data[(dset_key, i)] = prepare_table(hkl_table, crystal, symmetry)

In [ ]:
# calculate stats binned by resolution

SMAX = 0.8

def calc_bin_edges(bin_width):
    nbins = int(SMAX/bin_width)
    return np.linspace(0, nbins*bin_width, nbins + 1)

def calc_correlations(df, key1, key2, bin_width=0.01):
    df = df.dropna(subset=[key1, key2])
    # compute statistics in each bin
    bin_edges = calc_bin_edges(bin_width)
    s_bins = pd.cut(df["s"], bins=bin_edges)
    df_isoavg = df.groupby(s_bins, observed=False).agg(
        s=("s", "mean"),
        cc=(key1, lambda x: x.corr(df.loc[x.index, key2])),
    )
    return df_isoavg

def cc_to_ccstar(cc):
    cc_star = np.sqrt(2*cc/(cc+1))
    return cc_star

correlations = {}

for (key, part), df in data.items():
    print('Calculating cc 1/2 and cc* for dataset:',key)
    cc_all = calc_correlations(df, 'group_0_intensity_sub', 'group_1_intensity_sub')
    cc_excluded = calc_correlations(df[~df['is_reflection']], 'group_0_intensity_sub', 'group_1_intensity_sub')
    cc = cc_all.join(cc_excluded[['cc']], how='left', rsuffix='_excl')
    cc['cc_star'] = cc_to_ccstar(cc['cc'])
    cc['cc_excl_star'] = cc_to_ccstar(cc['cc_excl'])
    correlations[(key, part, part)] = cc

for (key1, part1), df1 in data.items():
    for (key2, part2), df2 in data.items():
        if (key1 != key2) or (part1 >= part2):
            continue
        print(f'Calculating cc and cc limit between parts {part1} and {part2} of dataset: {key1}')
        df12 = pd.merge(df1, df2, on=('h','k','l'), how='inner', suffixes=(None, '_2'))
        cc_all = calc_correlations(df12, 'intensity_sub', 'intensity_sub_2')
        cc_excluded = calc_correlations(df12[~df12['is_reflection']], 'intensity_sub', 'intensity_sub_2')
        cc = cc_all.join(cc_excluded[['cc']], how='left', rsuffix='_excl')
        cc1 = correlations[(key1, part1, part1)]
        cc2 = correlations[(key2, part2, part2)]
        cc['cc_limit'] = cc1['cc_star']*cc2['cc_star']
        cc['cc_excl_limit'] = cc1['cc_excl_star']*cc2['cc_excl_star']
        correlations[(key1, part1, part2)] = cc

In [ ]:
def add_row_header(ax, row_header, row_pad=3, rotate_row_headers=True, **text_kwargs):
    ax.annotate(
        row_header,
        xy=(0, 0.5),
        xytext=(-ax.yaxis.labelpad - row_pad, 0),
        xycoords=ax.yaxis.label,
        textcoords="offset points",
        ha="right",
        va="center",
        rotation=rotate_row_headers * 90,
        **text_kwargs,
    )

def add_col_header(ax, col_header, col_pad=7, **text_kwargs):
    ax.annotate(
        col_header,
        xy=(0.5, 1),
        xytext=(0, col_pad),
        xycoords="axes fraction",
        textcoords="offset points",
        ha="center",
        va="baseline",
        **text_kwargs,
    )

In [ ]:
# Version using data with Bragg excluded (_excl)

# sort datasets by date
ordered_dataset_keys = sorted(DATASETS.keys(), key=lambda x: x.split()[1])

for dataset_name in ordered_dataset_keys:
    # make a new plot for every dataset, where we have a 2x2 grid comparing parts 0 and 1. 
    # the diagonal entries are correlations[(dataset_name, 0, 0)] and correlations[(dataset_name, 1, 1)], 
    # while the off-diagonal entries are correlations[(dataset_name, 0, 1)] (skipping 1,0)

    # get the list of parts for this dataset
    parts = [part for (key, part) in data.keys() if key == dataset_name]

    fig, axs = plt.subplots(len(parts), len(parts), sharey=True, sharex=True, figsize=(5,5))

    
    for i, key1 in enumerate(parts):
        for j, key2 in enumerate(parts):
            df = correlations.get((dataset_name, key1, key2), correlations.get((dataset_name, key2, key1), None))
            if df is not None:
                if key1==key2:
                    df[['s','cc_excl_star']].set_index('s').plot(ax=axs[i][j],
                        legend=False,
                        color=['tab:purple'],
                        style=['-'],
                    )
                elif j > i: # above the diagonal
                    df[['s','cc_excl','cc_excl_limit']].set_index('s').plot(ax=axs[i][j],
                        legend=False,
                        color=['tab:blue','tab:cyan'],
                        style=['-','-'],
                    )
                    axs[i][j].fill_between(df["s"], df["cc_excl"], df["cc_excl_limit"], color="tab:cyan", alpha=0.2)
                else: # below the diagonal, skip it
                    axs[i][j].axis('off')
                    continue

    for j, key in enumerate(parts):
        axs[j][j].set_xlim([0,SMAX])
        axs[j][j].set_ylim([0,1.05])
        axs[j][j].set_xlabel('1/d (Å$^{-1}$)')
        axs[j][j].set_ylabel('Correlation coefficient')
        axs[j][j].tick_params(labelbottom=True, labelleft=True)
        axs[j][j].xaxis.label.set_visible(True)
        axs[j][j].yaxis.label.set_visible(True)
        add_row_header(axs[j][j], f'Part {key+1}', fontsize="large")
        add_col_header(axs[0][j], f'Part {key+1}', fontsize="large")

    fig.suptitle(f'{dataset_name}\nAnisotropic component\nBragg voxels excluded',fontsize="large")
    plt.tight_layout()
    plt.subplots_adjust(wspace=0.2, hspace=0.2)
    plt.show()
    plt.close()


In [ ]:
# Version using data with all voxels

# sort datasets by date
ordered_dataset_keys = sorted(DATASETS.keys(), key=lambda x: x.split()[1])

for dataset_name in ordered_dataset_keys:
    # make a new plot for every dataset, where we have a 2x2 grid comparing parts 0 and 1. 
    # the diagonal entries are correlations[(dataset_name, 0, 0)] and correlations[(dataset_name, 1, 1)], 
    # while the off-diagonal entries are correlations[(dataset_name, 0, 1)] (skipping 1,0)

    # get the list of parts for this dataset
    parts = [part for (key, part) in data.keys() if key == dataset_name]

    fig, axs = plt.subplots(len(parts), len(parts), sharey=True, sharex=True, figsize=(5,5))

    
    for i, key1 in enumerate(parts):
        for j, key2 in enumerate(parts):
            df = correlations.get((dataset_name, key1, key2), correlations.get((dataset_name, key2, key1), None))
            if df is not None:
                if key1==key2:
                    df[['s','cc_star']].set_index('s').plot(ax=axs[i][j],
                        legend=False,
                        color=['tab:purple'],
                        style=['-'],
                    )
                elif j > i: # above the diagonal
                    df[['s','cc','cc_limit']].set_index('s').plot(ax=axs[i][j],
                        legend=False,
                        color=['tab:blue','tab:cyan'],
                        style=['-','-'],
                    )
                    axs[i][j].fill_between(df["s"], df["cc"], df["cc_limit"], color="tab:cyan", alpha=0.2)
                else: # below the diagonal, skip it
                    axs[i][j].axis('off')
                    continue

    for j, key in enumerate(parts):
        axs[j][j].set_xlim([0,SMAX])
        axs[j][j].set_ylim([0,1.05])
        axs[j][j].set_xlabel('1/d (Å$^{-1}$)')
        axs[j][j].set_ylabel('Correlation coefficient')
        axs[j][j].tick_params(labelbottom=True, labelleft=True)
        axs[j][j].xaxis.label.set_visible(True)
        axs[j][j].yaxis.label.set_visible(True)
        add_row_header(axs[j][j], f'Part {key+1}', fontsize="large")
        add_col_header(axs[0][j], f'Part {key+1}', fontsize="large")

    fig.suptitle(f'{dataset_name}\nAnisotropic component\nAll voxels included',fontsize="large")
    plt.tight_layout()
    plt.subplots_adjust(wspace=0.2, hspace=0.2)
    plt.show()
    plt.close()


In [ ]:
# Dump the curves into csv files for reuse in manuscript figure

dataset_labels = {
    'ALS 2025-10-15': 'als_20251015_50ms',
    'ALS 2025-10-15 0.1s': 'als_20251015_100ms',
}

for (key, part1, part2), c in correlations.items():
    if key not in dataset_labels:
        continue
    if part1==part2:
        df = c[['s','cc_excl_star']].rename(columns={"cc_excl_star": "cc_star"})
        fn = Path('figures/source_data')/f"fig5_cc_star_{dataset_labels[key]}_part{part1+1}.csv"
    else:
        df = c[['s','cc_excl','cc_excl_limit']].rename(columns={"cc_excl": "cc", "cc_excl_limit": "cc_limit"})
        fn = Path('figures/source_data')/f"fig5_cc_{dataset_labels[key]}_part{part1+1}_vs_part{part2+1}.csv"
    df = df[(df['s']<=0.81) & (df['s']>0.03)] # trim resolution range
    df = df.set_index('s')
    print('Writing',fn)
    df.to_csv(fn)

## Compare map slices for low/high dose splits

In [ ]:
DATASETS

In [ ]:
# functions borrowed from compare_xtal_number_split.ipynb

from mdx2.report.visualization import calc_isoavg, extract_central_slice, unique_slices

def subtract_isotropic_part(da, s_iso, intensity_iso):
    s = np.sqrt(da.sx**2 + da.sy**2 + da.sz**2)
    out = da.copy()
    out.data = da.data - np.interp(s, s_iso, intensity_iso)
    return out

def ingest_merged_data(merged_nxs):
    crystal   = loadobj(merged_nxs, 'crystal')
    symmetry  = loadobj(merged_nxs, 'symmetry')
    hkl_table = loadobj(merged_nxs, 'hkl_table')
    s_iso, intensity_iso = calc_isoavg(hkl_table, crystal)
    slices = []
    for slice_index, coord in unique_slices(symmetry):
        intensity = extract_central_slice(hkl_table, symmetry, crystal,
            slice_index=slice_index, slice_coordinate=coord, signal="intensity")
        slices.append({'intensity': intensity,
                       'intensity_subtracted': subtract_isotropic_part(intensity, s_iso, intensity_iso)})
    return {'isotropic': {'s': s_iso, 'intensity': intensity_iso}, 'slices': slices}

bin_width = 0.01
smax = 0.8
nbins = int(smax / bin_width)
bin_edges = np.linspace(0, bin_width*nbins, nbins + 1)

def load_merged_calc_stats(merged_nxs):
    crystal   = loadobj(merged_nxs, 'crystal')
    symmetry  = loadobj(merged_nxs, 'symmetry')
    hkl_table = loadobj(merged_nxs, 'hkl_table')
    df = prepare_table(hkl_table, crystal, symmetry)
    # exclude Bragg points
    df = df[~df["is_reflection"]]
    # compute statistics in each bin
    def rmsavg(x):
        return np.sqrt(np.mean(x * x))
    s_bins = pd.cut(df["s"], bins=bin_edges)
    stats = df.groupby(s_bins, observed=False).agg(
        s=("s", "mean"),
        intensity_iso=("intensity_iso", "mean"),
        intensity_sub_rms=("intensity_sub", rmsavg),
        intensity_error_rms=("intensity_error", rmsavg),
    )
    return stats

In [ ]:
merged_data = {}

for dset_key, dset_roots in DATASETS.items():
    for i, dset_root in enumerate(dset_roots):
        merged_nxs = project_directory / dset_root / 'mdx2/merged.nxs'
        merged_data[(dset_key, i)] = ingest_merged_data(merged_nxs)

In [ ]:
#merged_data[('ALS 2025-10-15',0)]['slices']

for key in DATASETS:
    for slice_index in [1,3]:
        fig, ax = plt.subplots(1,2, figsize=(9, 4))
        for j, label in enumerate(['Part 1','Part 2']):
            vals = merged_data[(key,j)]
            normval = np.max(vals['isotropic']['intensity'])
            slice_data = vals['slices'][slice_index]
            s = np.sqrt(slice_data['intensity'].sx**2 + slice_data['intensity'].sy**2 + slice_data['intensity'].sz**2)
            #itot = slice_data['intensity'].where(s<0.8,np.nan)/normval
            isub = slice_data['intensity_subtracted'].where(s<0.8,np.nan)/normval
            #itot.plot.pcolormesh("sx", "sy", vmin=0, vmax=1.1, ax=ax[0])
            isub.plot.pcolormesh("sx", "sy", vmin=0.1, vmax=-0.1, cmap = 'RdBu_r', ax=ax[j])
            ax[j].set_title(label)
        for a in ax:
            a.set_xlim(-0.8, 0.8)
            a.set_ylim(-0.8, 0.8)
            a.set_aspect('equal', adjustable='box')
        dim_name = [name for name, coord in isub.coords.items() if coord.ndim == 0][0]
        dim_val = isub.coords[dim_name].values
        fig.suptitle(f"{key}\n{dim_name}={dim_val}")
        plt.tight_layout()
        fig.show()

In [ ]:
# Dump plot data for Figure 5
#
# need: parts 1, 2 for 0.05s data, and same for 0.1s data

import xarray as xr
import json

dataset_labels = {
    'ALS 2025-10-15': 'als_20251015_50ms',
    'ALS 2025-10-15 0.1s': 'als_20251015_100ms',
}

slice_index=1
for key, dset_label in dataset_labels.items():
    slices = []
    for j, part_label in enumerate(['Part 1','Part 2']):
        vals = merged_data[(key,j)]
        normval = np.max(vals['isotropic']['intensity'])
        sd = vals['slices'][slice_index]['intensity_subtracted']
        sd = sd/normval
        sd = sd.astype('float32')
        sd = sd.rename(part_label)
        if j>0:
             sd = sd.drop_vars(('sx','sy','sz'))
        slices.append(sd)
    da = xr.merge(slices)
    s = np.sqrt(da.sx**2 + da.sy**2 + da.sz**2)
    da = da.where(s < 0.8, np.nan)
    da = da.where(da['Part 1'].notnull(),drop=True)
    fn = f'figures/source_data/fig5_slices_{dset_label}.json'
    with open(fn,'w') as f:
        print('writing',fn)
        json.dump(da.to_dict(),f)

In [ ]:
merged_stats = {}

for dset_key, dset_roots in DATASETS.items():
    for i, dset_root in enumerate(dset_roots):
        merged_nxs = project_directory / dset_root / 'mdx2/merged.nxs'
        print('loading',merged_nxs)
        df = load_merged_calc_stats(merged_nxs)
        df['signal_var_est'] = df['intensity_sub_rms']**2 - df['intensity_error_rms']**2
        df.loc[df["signal_var_est"] < 0, "signal_var_est"] = np.nan 
        df.loc[df['s'] < 0.04,'signal_var_est'] = np.nan
        df['irmscorr'] = np.sqrt(df['signal_var_est'])
        merged_stats[(dset_key, i)] = df

In [ ]:
for key in DATASETS:
    fig, axs = plt.subplots(1,2,figsize=(7,3),sharex=True)
    df1 = merged_stats[(key,0)]
    df2 = merged_stats[(key,1)]
    for df, label in zip([df1,df2],['Part 1','Part 2']):
        df.plot(x='s',y='irmscorr',ax=axs[1],label=label)
        df.plot(x='s',y='intensity_iso',ax=axs[0],label=label)
    axs[0].set_title('Isotropic')
    axs[1].set_title('Ansotropic')
    fig.suptitle(key)
    plt.tight_layout()
    fig.show()

In [ ]:
# dump to csv for figure making

dataset_labels = {
    'ALS 2025-10-15': 'als_20251015_50ms',
    'ALS 2025-10-15 0.1s': 'als_20251015_100ms',
}

for key, label in dataset_labels.items():
    fn = f'figures/source_data/fig5_iso_aniso_{label}.csv'
    df1 = merged_stats[(key,0)][['s','intensity_iso','irmscorr']]
    df2 = merged_stats[(key,1)][['intensity_iso','irmscorr']]
    df = df1.join(df2, lsuffix='_1', rsuffix='_2').set_index('s')
    df/=df['intensity_iso_1'].max() # normalize all columns by iso max of first half
    print('Writing',fn)
    df.to_csv(fn)